In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [2]:
import json
import random
import os
from datetime import datetime, timedelta

os.makedirs('data', exist_ok=True)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']
start_time = datetime.now().replace(hour=8, minute=0, second=0, microsecond=0)
total_records = 10000

print("Generowanie poprawionego pliku data/transactions_10k.jsonl...")

with open('data/transactions_10k.jsonl', 'w', encoding='utf-8') as f:
    for i in range(total_records):
        timestamp = start_time + timedelta(seconds=i * 1.08)
        
        tx = {
            'tx_id': f'TX{1000 + i}',
            'user_id': f'u{random.randint(1, 20):02d}',
            'amount': round(random.uniform(5.0, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'timestamp': timestamp.strftime("%Y-%m-%d %H:%M:%S")
        }
        
        f.write(json.dumps(tx) + '\n')

print("Gotowe! Teraz wczytaj plik w Sparku – powinno przejść bez błędu.")

Generowanie poprawionego pliku data/transactions_10k.jsonl...
Gotowe! Teraz wczytaj plik w Sparku – powinno przejść bez błędu.


In [3]:
#df = spark.read.json("data/transactions_10k.jsonl")
df = spark.read.option("inferSchema", "true").json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
df.show(10, truncate=False)

+-------+-----------+--------+-------------------+------+-------+
|amount |category   |store   |timestamp          |tx_id |user_id|
+-------+-----------+--------+-------------------+------+-------+
|1884.15|żywność    |Gdańsk  |2026-05-10 08:00:00|TX1000|u10    |
|4559.26|książki    |Wrocław |2026-05-10 08:00:01|TX1001|u14    |
|213.35 |książki    |Warszawa|2026-05-10 08:00:02|TX1002|u03    |
|3678.58|odzież     |Kraków  |2026-05-10 08:00:03|TX1003|u09    |
|1432.07|żywność    |Wrocław |2026-05-10 08:00:04|TX1004|u17    |
|2817.83|żywność    |Gdańsk  |2026-05-10 08:00:05|TX1005|u07    |
|4606.66|elektronika|Warszawa|2026-05-10 08:00:06|TX1006|u07    |
|4218.94|książki    |Kraków  |2026-05-10 08:00:07|TX1007|u03    |
|1839.1 |książki    |Wrocław |2026-05-10 08:00:08|TX1008|u13    |
|2806.84|książki    |Kraków  |2026-05-10 08:00:09|TX1009|u15    |
+-------+-----------+--------+-------------------+------+-------+
only showing top 10 rows



In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2468|6213026.59|    2517.43|
|  Kraków|     2573|6461557.34|    2511.29|
|Warszawa|     2498|6107221.23|    2444.84|
| Wrocław|     2461|6168328.32|    2506.43|
+--------+---------+----------+-----------+



In [7]:
# Policz sumę, minimum i maksimum kwoty dla każdej kategorii.

from pyspark.sql.functions import min as _min, max as _max, sum as _sum, round as _round

# TWÓJ KOD
# df.groupBy("category").agg(...).orderBy("category").show()

cat_min_max = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _min("amount").alias("min_PLN"),
        _max("amount").alias("max_PLN")
    )
    .orderBy("category")
)

In [8]:
cat_min_max.show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|6243226.55|   6.53|4998.83|
|    książki|6255624.61|   8.82|4999.13|
|     odzież|6124572.52|   5.34|4997.77|
|    żywność| 6326709.8|   6.92| 4999.9|
+-----------+----------+-------+-------+



In [9]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-05-10 08:00:00, 2026-05-10 09:00:00}|3334     |8290488.73|
|{2026-05-10 09:00:00, 2026-05-10 10:00:00}|3333     |8277730.05|
|{2026-05-10 10:00:00, 2026-05-10 11:00:00}|3333     |8381914.7 |
+------------------------------------------+---------+----------+



In [10]:
#Policz transakcje i sumę per sklep w każdym 30-minutowym oknie. Posortuj po oknie, a w ramach okna po sklepie.

# TWÓJ KOD
# df.groupBy(window("timestamp", "30 minutes"), "store").agg(...).orderBy(...).show()

from pyspark.sql.functions import window, count, sum as _sum, round as _round

hourly1 = (
    df.groupBy(
        window("timestamp", "30 minutes"), 
        "store"
    )
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy("window", "store")
)

hourly1.show(truncate=False)

+------------------------------------------+--------+---------+----------+
|window                                    |store   |liczba_tx|suma_PLN  |
+------------------------------------------+--------+---------+----------+
|{2026-05-10 08:00:00, 2026-05-10 08:30:00}|Gdańsk  |426      |1083347.84|
|{2026-05-10 08:00:00, 2026-05-10 08:30:00}|Kraków  |452      |1116596.55|
|{2026-05-10 08:00:00, 2026-05-10 08:30:00}|Warszawa|403      |997337.37 |
|{2026-05-10 08:00:00, 2026-05-10 08:30:00}|Wrocław |386      |986132.68 |
|{2026-05-10 08:30:00, 2026-05-10 09:00:00}|Gdańsk  |405      |1006229.89|
|{2026-05-10 08:30:00, 2026-05-10 09:00:00}|Kraków  |424      |1110830.61|
|{2026-05-10 08:30:00, 2026-05-10 09:00:00}|Warszawa|420      |980510.44 |
|{2026-05-10 08:30:00, 2026-05-10 09:00:00}|Wrocław |418      |1009503.35|
|{2026-05-10 09:00:00, 2026-05-10 09:30:00}|Gdańsk  |389      |982755.25 |
|{2026-05-10 09:00:00, 2026-05-10 09:30:00}|Kraków  |425      |1049534.72|
|{2026-05-10 09:00:00, 20

In [11]:
# W której godzinie sklep “Kraków” miał najwyższy przychód?
# Filtruj najpierw po sklepie, potem zrób okno godzinne, posortuj malejąco po sumie.

# TWÓJ KOD

from pyspark.sql.functions import window, count, sum as _sum, round as _round, desc

krak_max = (
    df.filter(df["store"] == "Kraków")
    .groupBy("store", window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy(desc("suma_PLN"))
)

krak_max.show(truncate=False)

+------+------------------------------------------+---------+----------+
|store |window                                    |liczba_tx|suma_PLN  |
+------+------------------------------------------+---------+----------+
|Kraków|{2026-05-10 08:00:00, 2026-05-10 09:00:00}|876      |2227427.16|
|Kraków|{2026-05-10 09:00:00, 2026-05-10 10:00:00}|863      |2147844.21|
|Kraków|{2026-05-10 10:00:00, 2026-05-10 11:00:00}|834      |2086285.97|
+------+------------------------------------------+---------+----------+



In [12]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-10 07:30:00|2026-05-10 08:30:00|1667     |4183414.44|
|2026-05-10 08:00:00|2026-05-10 09:00:00|3334     |8290488.73|
|2026-05-10 08:30:00|2026-05-10 09:30:00|3333     |8233146.9 |
|2026-05-10 09:00:00|2026-05-10 10:00:00|3333     |8277730.05|
|2026-05-10 09:30:00|2026-05-10 10:30:00|3334     |8328947.57|
|2026-05-10 10:00:00|2026-05-10 11:00:00|3333     |8381914.7 |
|2026-05-10 10:30:00|2026-05-10 11:30:00|1666     |4204624.57|
+-------------------+-------------------+---------+----------+



In [13]:
#Zadanie 4.2 — Porównaj tumbling vs sliding
#Policz łączną liczbę wierszy wynikowych w obu podejściach. Dlaczego sliding daje więcej wierszy?

tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:

# Sliding window generuje większą liczbę wierszy, ponieważ okna czasowe nachodzą na siebie. 
# Przy parametrach (1h duration, 30min slide), każde zdarzenie jest procesowane dwukrotnie, 
# gdyż wpada do dwóch sąsiednich okien jednocześnie - w przeciwieństwie do okien typu Tumbling, 
# które są rozłączne.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [14]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ:
#    3333 (na podstawie stworzonego pliku transactions_10k.jsonl)

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ:
#    groupBy("store") wykonuje agregację globalną dla każdego sklepu (jeden wynik 
#    na sklep dla całego zbioru danych). 
#    groupBy(window(...), "store") wykonuje agregację w podziale na czas, czyli 
#    oblicza statystyki osobno dla każdego przedziału czasowego (np. co godzinę).

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ:
#    Zawierają ją 2 okna. Transakcja z godziny 09:30 wpada do okna, które 
#    zaczyna się o 09:00 oraz do okna, które zaczyna się o 09:30.

# Zadania domowe

In [15]:
from pyspark.sql.functions import window, count, sum as _sum, avg, round as _round, col, desc, asc, to_timestamp, lit

gdansk_min_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN"),
        count("tx_id").alias("liczba_tx"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "srednia_PLN",
    )
    .orderBy(asc("srednia_PLN"))
)
gdansk_min_avg.limit(1).show(truncate=False)

window_start = "2024-01-01 09:00:00"
window_end   = "2024-01-01 09:30:00"

kategorie_okno = (
    df.filter(
        (col("timestamp") >= to_timestamp(lit(window_start), "yyyy-MM-dd HH:mm:ss")) &
        (col("timestamp") <  to_timestamp(lit(window_end),   "yyyy-MM-dd HH:mm:ss"))
    )
    .groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy(desc("liczba_tx"))
)
kategorie_okno.show(truncate=False)

szczyt_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy(desc("liczba_tx"))
)
szczyt_15min.limit(1).show(truncate=False)

+-------------------+-------------------+---------+-----------+
|od                 |do                 |liczba_tx|srednia_PLN|
+-------------------+-------------------+---------+-----------+
|2026-05-10 09:00:00|2026-05-10 10:00:00|814      |2493.38    |
+-------------------+-------------------+---------+-----------+

+--------+---------+--------+
|category|liczba_tx|suma_PLN|
+--------+---------+--------+
+--------+---------+--------+

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-10 09:30:00|2026-05-10 09:45:00|834      |2058305.98|
+-------------------+-------------------+---------+----------+

